# Data Exploration Notebook

This notebook helps profile the **raw datar** and define the **transformations**.


In [1]:
%pip install pandas

import pandas as pd
import os

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1) Select raw data

In [2]:
folder_path = "../raw/"
raw_path = os.listdir(folder_path)
raw_data_path = ""
last_ingestion = 0

for filename in raw_path:
	if "jobs_" in filename and filename.endswith(".jsonl"):
		date_part  = filename.split("jobs_")[-1].split(".jsonl")[0]
		max_date = int(date_part.replace("_", ""))
		if max_date > last_ingestion:
		 	raw_data_path = filename

if not raw_data_path:
	raise ValueError("Raw data path not found.")
else:
	final_path = folder_path + raw_data_path
	df = pd.read_json(final_path, lines=True)
	# Initial conferences
	print(f"Rows count: {len(df):,}")
	print(f"Columns count: {len(df.columns)}")
	print(f"Columns: {df.columns}")


Rows count: 701
Columns count: 5
Columns: Index(['source', 'search_keyword', 'search_location', 'scraped_at', 'raw'], dtype='object')


In [3]:
# Transform 'raw' column dict in DataFrame
df_raw_columns = pd.json_normalize(df['raw'])
df = pd.concat([df, df_raw_columns], axis=1)
df = df.drop(columns=['raw'])

df.columns


Index(['source', 'search_keyword', 'search_location', 'scraped_at', 'id',
       'site', 'job_url', 'job_url_direct', 'title', 'company', 'location',
       'date_posted', 'job_type', 'salary_source', 'interval', 'min_amount',
       'max_amount', 'currency', 'is_remote', 'job_level', 'job_function',
       'listing_type', 'emails', 'description', 'company_industry',
       'company_url', 'company_logo', 'company_url_direct',
       'company_addresses', 'company_num_employees', 'company_revenue',
       'company_description', 'skills', 'experience_range', 'company_rating',
       'company_reviews_count', 'vacancy_count', 'work_from_home_type'],
      dtype='object')

## 2) Basic schema profiling

In [4]:
schema_df = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "non_null": [int(df[c].notna().sum()) for c in df.columns],
    "nulls": [int(df[c].isna().sum()) for c in df.columns],
    "null_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [int(df[c].nunique(dropna=True)) for c in df.columns],
}).sort_values(["null_pct", "column"], ascending=[False, True])

schema_df


,column,dtype,non_null,nulls,null_pct,n_unique
34,company_rating,object,0,701,100.00,0
35,company_reviews_count,object,0,701,100.00,0
17,currency,object,0,701,100.00,0
33,experience_range,object,0,701,100.00,0
14,interval,object,0,701,100.00,0
21,listing_type,object,0,701,100.00,0
16,max_amount,object,0,701,100.00,0
15,min_amount,object,0,701,100.00,0
13,salary_source,object,0,701,100.00,0
32,skills,object,0,701,100.00,0


In [5]:
# Check the need to be normalized in Silver
key_dims = [c for c in df.columns]
for col in key_dims:
    print(f"\n=== {col} ===")
    display(
        df[col]
        .astype("string")
        .str.strip()
        .fillna("<NULL>")
        .value_counts(dropna=False)
        .head(10)
        .to_frame("count")
    )



=== source ===


,count
source,
linkedin,360
indeed,341



=== search_keyword ===


,count
search_keyword,
data engineer,443
ai data engineer,258



=== search_location ===


...

,count
company_reviews_count,
<NULL>,701



=== vacancy_count ===


,count
vacancy_count,
<NULL>,701



=== work_from_home_type ===


,count
work_from_home_type,
<NULL>,701


## 3) Duplicate and key profiling

In [6]:
if "id" in df.columns:
    dup_count = int(df.duplicated(subset=["id", "source"]).sum())
    print("Duplicated id per source:", dup_count)

    if dup_count:
        display(df[df.duplicated(subset=["id", "source"], keep=False)].sort_values("id").head(10))
else:
    print("Column 'id' not found.")


Duplicated id per source: 316


,source,search_keyword,search_location,scraped_at,id,site,job_url,job_url_direct,title,company,...,company_addresses,company_num_employees,company_revenue,company_description,skills,experience_range,company_rating,company_reviews_count,vacancy_count,work_from_home_type
394,indeed,data engineer,"Sydney, Australia",2026-06-20 22:09:03.642112+00:00,in-00e3024154fb2516,indeed,https://au.indeed.com/viewjob?jk=00e3024154fb2516,https://iworkfor.nsw.gov.au/job/engineer-584213,Engineer,NSW Government,...,Sydney,"10,000+",None,None,None,None,None,None,None,None
560,indeed,data engineer,Australia,2026-06-20 22:09:12.879996+00:00,in-00e3024154fb2516,indeed,https://au.indeed.com/viewjob?jk=00e3024154fb2516,https://iworkfor.nsw.gov.au/job/engineer-584213,Engineer,NSW Government,...,Sydney,"10,000+",None,None,None,None,None,None,None,None
627,indeed,ai data engineer,"Sydney, Australia",2026-06-20 22:09:15.483358+00:00,in-01492af9a249ba09,indeed,https://au.indeed.com/viewjob?jk=01492af9a249ba09,https://h2oai./apply/h6ZS1I79Ik/AI-Engineer?so...,AI Engineer,H2O.ai,...,Mountain View,11 to 50,less than $1M (USD),None,None,None,None,None,None,None
693,indeed,ai data engineer,Australia,2026-06-20 22:09:22.096130+00:00,in-01492af9a249ba09,indeed,https://au.indeed.com/viewjob?jk=01492af9a249ba09,https://h2oai./apply/h6ZS1I79Ik/AI-Engineer?so...,AI Engineer,H2O.ai,...,Mountain View,11 to 50,less than $1M (USD),None,None,None,None,None,None,None
421,indeed,data engineer,"Sydney, Australia",2026-06-20 22:09:03.642112+00:00,in-01492af9a249ba09,indeed,https://au.indeed.com/viewjob?jk=01492af9a249ba09,https://h2oai./apply/h6ZS1I79Ik/AI-Engineer?so...,AI Engineer,H2O.ai,...,Mountain View,11 to 50,less than $1M (USD),None,None,None,None,None,None,None
584,indeed,data engineer,Australia,2026-06-20 22:09:12.879996+00:00,in-01492af9a249ba09,indeed,https://au.indeed.com/viewjob?jk=01492af9a249ba09,https://h2oai./apply/h6ZS1I79Ik/AI-Engineer?so...,AI Engineer,H2O.ai,...,Mountain View,11 to 50,less than $1M (USD),None,None,None,None,None,None,None
407,indeed,data engineer,"Sydney, Australia",2026-06-20 22:09:03.642112+00:00,in-026ef088eca5582a,indeed,https://au.indeed.com/viewjob?jk=026ef088eca5582a,https://grnh.se/cmlqx9rn4us,Lead Software Engineer,Karbon,...,United States\r\nCanada\r\nNew Zealand\r\nAust...,51 to 200,None,We’re on a mission to change the way accountin...,None,None,None,None,None,None
683,indeed,ai data engineer,Australia,2026-06-20 22:09:22.096130+00:00,in-026ef088eca5582a,indeed,https://au.indeed.com/viewjob?jk=026ef088eca5582a,https://grnh.se/cmlqx9rn4us,Lead Software Engineer,Karbon,...,United States\r\nCanada\r\nNew Zealand\r\nAust...,51 to 200,None,We’re on a mission to change the way accountin...,None,None,None,None,None,None
619,indeed,ai data engineer,"Sydney, Australia",2026-06-20 22:09:15.483358+00:00,in-026ef088eca5582a,indeed,https://au.indeed.com/viewjob?jk=026ef088eca5582a,https://grnh.se/cmlqx9rn4us,Lead Software Engineer,Karbon,...,United States\r\nCanada\r\nNew Zealand\r\nAust...,51 to 200,None,We’re on a mission to change the way accountin...,None,None,None,None,None,None
671,indeed,ai data engineer,Australia,2026-06-20 22:09:22.096130+00:00,in-04d2eb56fba078d5,indeed,https://au.indeed.com/viewjob?jk=04d2eb56fba078d5,https://jobs.lever.co/immuta/d56a5011-8fbd-4b1...,Solutions Engineer,Immuta,...,College Park,51 to 200,None,None,None,None,None,None,None,None
